# QLoRA Fine-Tuning for HTML → JSON Extraction

This demo fine-tunes a small open model (**Qwen3-4B-Instruct-2507**) with **QLoRA** to
turn product HTML snippets into structured JSON. QLoRA means the base model is loaded in
4-bit and only small LoRA adapters are trained, so the whole run fits on a single modest
GPU and finishes in a few minutes. The result is exported as a **GGUF** file you can
run locally with Ollama / llama.cpp.

The pipeline is: load the 500-example dataset → load the 4-bit base model → attach LoRA
adapters → supervised fine-tune → test → export to GGUF.

## 1. Load the dataset

Make sure `json_extraction_dataset_500.json` is in the working directory before running
this cell.

In [2]:
import json

file = json.load(open("json_extraction_dataset_500.json"))
print(f"Loaded {len(file)} examples")
print(json.dumps(file[1], indent=2))

Loaded 500 examples
{
  "input": "Extract the product information:\n<div class='product'><h2>iPad Air</h2><span class='price'>$1344</span><span class='category'>audio</span><span class='brand'>Dell</span></div>",
  "output": {
    "name": "iPad Air",
    "price": "$1344",
    "category": "audio",
    "manufacturer": "Dell"
  }
}


Each example has an `input` (the raw HTML) and an `output` (the JSON we want).
The four keys are always `name`, `price`, `category` and `manufacturer`.

## 2. Install dependencies

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Local / cloud setups
else:
    import torch
    v = re.match(r"[\d]{1,}\.[\d]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + {"2.10": "0.0.34", "2.9": "0.0.33.post1", "2.8": "0.0.32.post2"}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

## 3. Check the environment

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

## 4. Load the base model in 4-bit (QLoRA)

QLoRA starts by quantizing the frozen base model to 4-bit, which cuts its memory use
roughly 4× and lets a 4B model fit on modest hardware. Under the hood Unsloth uses 4-bit
**NF4** quantization with double quantization, equivalent to a `BitsAndBytesConfig` with
`bnb_4bit_quant_type="nf4"` and `bnb_4bit_use_double_quant=True`. Only the LoRA adapters we
add later are trained, and those stay in higher precision. `get_chat_template`
installs the exact ChatML template the model expects so training and inference use the
*same* prompt format.

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

model_name = "unsloth/Qwen3-4B-Instruct-2507"
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=True,  # QLoRA: quantize the frozen base model to 4-bit
)

# Use the model's native chat template so training and inference match.
tokenizer = get_chat_template(tokenizer, chat_template="qwen3-instruct")

## 5. Build the training text

We turn every example into a 3-message conversation (system → user → assistant) and
render it through the chat template. Training on the rendered chat text means the model
learns the same format we will use at inference time.

In [ ]:
from datasets import Dataset

SYSTEM_PROMPT = (
    "You extract product information from HTML and reply with a single JSON object "
    'using the keys "name", "price", "category" and "manufacturer".'
)

def format_prompt(example):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["input"]},
        {"role": "assistant", "content": json.dumps(example["output"])},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False)

dataset = Dataset.from_dict({"text": [format_prompt(row) for row in file]})
print(dataset[0]["text"])

## 6. Attach LoRA adapters (the LoRA half of QLoRA)

Instead of updating all ~4B weights, LoRA freezes the 4-bit base model and trains a tiny
set of low-rank matrices added to the attention and MLP layers. Together — a 4-bit base
model plus LoRA adapters — this is QLoRA, and it is what makes fine-tuning cheap enough
for a modest GPU while producing a small adapter file.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

## 7. Configure the trainer

`SFTTrainer` runs supervised fine-tuning. Key settings:

| Setting | Value | Why |
|---|---|---|
| `per_device_train_batch_size` | 2 | Fits GPU memory |
| `gradient_accumulation_steps` | 4 | Effective batch size = 8 |
| `num_train_epochs` | 3 | Dataset is small (500 rows) |
| `learning_rate` | 2e-4 | Typical QLoRA learning rate |
| `optim` | `adamw_8bit` | Memory-efficient optimizer |
| `bf16`/`fp16` | auto | Fast mixed precision on the GPU |

`train_on_responses_only` masks the prompt tokens so the loss is computed **only on the
JSON answer** — the model is not rewarded for reproducing the question.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_length=max_seq_length,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,  # effective batch size = 8
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        logging_steps=25,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        save_strategy="epoch",
        save_total_limit=2,
        output_dir="outputs",
        seed=3407,
        report_to="none",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
    ),
)

# Only compute the loss on the assistant's JSON answer.
trainer = train_on_responses_only(trainer)

## 8. Train

In [ ]:
trainer_stats = trainer.train()
print(f"Training time: {trainer_stats.metrics['train_runtime'] / 60:.2f} minutes")
print(f"Peak VRAM: {torch.cuda.max_memory_reserved() / 1024**3:.2f} GB")

## 9. Test the fine-tuned model

We use the *same* chat template as training, with `add_generation_prompt=True` so the
model starts generating the assistant turn. Qwen3-Instruct recommends
`temperature=0.7, top_p=0.8, top_k=20`.

In [ ]:
from transformers import TextStreamer

FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Extract the product information:\n<div class='product'><h2>iPad Air</h2><span class='price'>$1344</span><span class='category'>audio</span><span class='brand'>Dell</span></div>"},
]

inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.8,
    top_k=20,
    streamer=TextStreamer(tokenizer, skip_prompt=True),
)

print("\nFinal JSON:", tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True).strip())

## 10. Export to GGUF

GGUF is the format used by Ollama and llama.cpp. `q4_k_m` is a good accuracy/size
trade-off. The model is written to the `gguf_model_gguf/` folder, and you can copy the `.gguf`
out of there when you want to run it locally.

In [ ]:
# Export to GGUF (q4_k_m) for Ollama / llama.cpp.
# The .gguf is written to the "gguf_model_gguf/" folder.
model.save_pretrained_gguf("gguf_model", tokenizer, quantization_method="q4_k_m")

## 11. Run it locally with Ollama

Copy the exported `.gguf` into this repo's folder and point the `Modelfile` at it,
then:

```bash
ollama create qwen3-json-extractor -f Modelfile
ollama run qwen3-json-extractor "Extract the product information:
<div class='product'><h2>iPad Air</h2><span class='price'>$1344</span><span class='category'>audio</span><span class='brand'>Dell</span></div>"
```

See `README.md` for a full explanation of how the demo works.